<div align="center">

# 🔬 HistoMoE: Interactive Inference Demo

This notebook demonstrates how to load a trained **HistoMoE** model, pass a histology image patch through it, and visualize the resulting expert routing weights and gene expression predictions.

</div>

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure the histomoe package is in the path
sys.path.insert(0, os.path.abspath('..'))

from histomoe.models.histomoe_model import HistoMoE
from histomoe.data.metadata_utils import CANCER_TYPES, cancer_type_to_id
from histomoe.data.transforms import get_transforms
from histomoe.visualization.routing_viz import plot_routing_weights

### 1. Initialize the Model
Normally, you would load a trained checkpoint here using `HistoMoE.load_from_checkpoint("path/to/best.ckpt")`. For this demo, we will instantiate a fresh model with random weights to demonstrate the API.

In [ ]:
N_GENES = 250
N_EXPERTS = 5
BACKBONE = "resnet50"

print(f"Initializing HistoMoE with {BACKBONE} backbone...")
model = HistoMoE(
    backbone=BACKBONE,
    n_genes=N_GENES,
    n_experts=N_EXPERTS,
    gating_mode="soft",
    pretrained_backbone=False  # Fast loading for demo
)
model.eval()
print("Model loaded successfully!")

### 2. Prepare Input Data
We need a histology image patch and a cancer type label. We will create a synthetic image here, but you can replace `image_np` with `np.array(Image.open("your_patch.png"))`.

In [ ]:
# Simulate a random RGB histology patch [H, W, C]
image_np = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
image_pil = Image.fromarray(image_np)

# Target cancer type
target_cancer = "LUAD"
cancer_id = cancer_type_to_id(target_cancer)

print(f"Cancer Type: {target_cancer} (ID: {cancer_id})")

plt.imshow(image_pil)
plt.title(f"Synthetic Histology Patch ({target_cancer})")
plt.axis('off')
plt.show()

### 3. Run Inference
We apply the test transformations and pass the data through the model.

In [ ]:
transform = get_transforms(split="test", patch_size=224)
image_tensor = transform(image_pil).unsqueeze(0)  # [1, 3, 224, 224]
cancer_tensor = torch.tensor([cancer_id], dtype=torch.long)

with torch.no_grad():
    result = model.predict_patches(image_tensor, cancer_tensor)

predictions = result["predictions"].numpy()
routing_weights = result["routing_weights"].numpy()
dominant_expert = result["dominant_expert"].numpy()[0]

print(f"Predicted expression shape: {predictions.shape}")
print(f"Routing weights shape: {routing_weights.shape}")
print(f"Dominant Expert: {CANCER_TYPES[dominant_expert]}")

### 4. Visualize Expert Routing
Let's see how much each expert contributed to the final prediction.

In [ ]:
# Plot routing weights using the visualization tool
plot_routing_weights(
    routing_weights,
    cancer_types=CANCER_TYPES,
    title=f"Routing Distribution for {target_cancer} Patch"
)